# Superstore - Retail

In [1]:
import pandas as pd 
import numpy as np 

In [2]:
data = pd.read_csv("../data/raw/superstore-tableau.csv")

In [4]:
data.shape

(9994, 21)

In [3]:
data.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,08-11-2016,11-11-2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,08-11-2016,11-11-2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,12-06-2016,16-06-2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,11-10-2015,18-10-2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,11-10-2015,18-10-2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [ ]:
data.columns

In [11]:
# Creating unique data tables
data["product_id"] = [x.split("-")[2] for x in data["Product ID"].values]
customer = data[["Customer ID", "Customer Name", "Segment", "Country", "State", "City", "Postal Code", "Region"]].drop_duplicates()
product = data[["product_id", "Category", "Sub-Category", "Product Name"]].drop_duplicates()
orders = data[["Customer ID", "Order ID", "Order Date", "Ship Date", "Ship Mode", "Product ID", "Sales", "Quantity", "Discount", "Profit"]].drop_duplicates()
orders['Order Date'] = pd.to_datetime(orders['Order Date'], format = "mixed")
orders['Ship Date'] = pd.to_datetime(orders['Ship Date'], format = "mixed")
orders["days_till_shipped"] = (orders["Ship Date"] - orders["Order Date"]).dt.days

dates = pd.DataFrame(orders['Order Date'].unique()).sort_values(by=0).reset_index(drop=True)
dates.columns = ['date']
dates['month'] = dates['date'].dt.month_name()
dates['month_num'] = dates['date'].dt.month
dates['year'] = dates['date'].dt.year
dates['quarter'] = dates['date'].dt.quarter


product_price_history = data[["product_id", "Product ID","Category","Sub-Category", "Discount", "Sales","Quantity",
        "Profit"]].drop_duplicates().merge(orders, 
                        on="Product ID").merge(
                            dates, 
                            left_on="Order Date", 
                            right_on="date")[["product_id", "Category","Sub-Category",
                                                "year","month_num", "month",
                                                "Discount_x",
                                                "Sales_x","Quantity_x","Profit_x"]].drop_duplicates().sort_values(
                     ["year","month_num", "product_id"]).reset_index(drop=True)


product_price = product_price_history[product_price_history["Discount_x"] == 0.0]
product_price["Sales_x"] = round(product_price["Sales_x"] / product_price_history["Quantity_x"],2)
product_price["Profit_x"] = round(product_price["Profit_x"] / product_price_history["Quantity_x"],2)
product_price["cost"] = round(product_price["Sales_x"] - product_price["Profit_x"],2)
product_price= product_price[["product_id", "Category","Sub-Category","year","month","cost"]].drop_duplicates().reset_index(drop=True)
product_price.columns = ["product_id","category","sub_category","year","month","cost"]

product_price_disc = product_price_history[product_price_history["Discount_x"] != 0.0]
product_price_disc["cost"] = round(product_price_disc["Sales_x"] - product_price_disc["Profit_x"],2)
product_price_disc= product_price_disc[["product_id","year","month", "Sales_x","Discount_x", "Profit_x","cost"]].drop_duplicates().reset_index(drop=True)
product_price_disc.columns = ["product_id","year","month","revenue", "discount", "profit","cost"]



In [15]:
# creating aggregated view of orders
columns = ['Order ID', 'Order Date', 'year',"quarter",  'month', "month_num",'Ship Mode', 'days_till_shipped', 'Sales',
       'Quantity', 'Product ID' ]
orders_agg = orders.groupby(["Order ID", 
                             "Order Date", 
                             "Ship Mode",
                             "days_till_shipped"])[
                                 ["Sales",
                                  "Quantity"]
                                 ].sum().reset_index().merge(
                                     orders.groupby("Order ID")["Product ID"].count(),
                                      on="Order ID").merge(dates[["date",
                                                        "year","quarter", "month","month_num"]], 
                                                        left_on="Order Date", 
                                                        right_on = "date")[columns]
orders_agg = orders_agg.sort_values('Order Date').reset_index(drop=True)
orders_agg.columns = ['order_id', 'order_date', 'year',"quarter",'month', "month_num",'ship_mode', 'days_till_shipped', 'revenue',
       'quantity', 'products' ]
orders_agg

,order_id,order_date,year,quarter,month,month_num,ship_mode,days_till_shipped,revenue,quantity,products
0,CA-2014-140795,2014-01-02,2014,1,January,1,First Class,59,468.900,6,1
1,CA-2014-104269,2014-01-03,2014,1,January,1,Second Class,151,457.568,2,1
2,CA-2014-168312,2014-01-03,2014,1,January,1,Standard Class,181,513.861,6,2
3,CA-2014-113880,2014-01-03,2014,1,January,1,Standard Class,120,651.588,9,2
4,US-2014-143707,2014-01-03,2014,1,January,1,Standard Class,120,5.940,3,1
...,...,...,...,...,...,...,...,...,...,...,...
5004,US-2017-158526,2017-12-29,2017,4,December,12,Second Class,3,1814.680,14,5
5005,CA-2017-156720,2017-12-30,2017,4,December,12,Standard Class,61,3.024,3,1
5006,CA-2017-143259,2017-12-30,2017,4,December,12,Standard Class,61,466.842,14,3
5007,CA-2017-115427,2017-12-30,2017,4,December,12,Standard Class,61,34.624,4,2


In [16]:
# Let's check total  orders
print("Total orders:", len(orders_agg['order_id']))

# Let's check total  sales
print("Total sales:", round(orders_agg['revenue'].sum(), 2))

# Let's check total  customers
print("Total customers:", len(orders['Customer ID'].unique()))


Total orders: 5009
Total sales: 2296919.49
Total customers: 793


## Business Goal - Increase revenue

Are sales growing every year?

In [17]:
# Total orders, sales, products, quantities through 2014-2017
orders_rev = (orders_agg.groupby(["year"])["order_id"].count().reset_index()).join(
    orders_agg.groupby(["year"])["products"].sum(), 
    how="left", 
    on="year").join(
    orders_agg.groupby(["year"])["quantity"].sum(), 
    how="left", 
    on="year").join(
    orders_agg.groupby(["year"])["revenue"].sum(), 
    how="left", 
    on="year")
orders_rev["%_change"] = round(orders_rev["revenue"].pct_change(),2)
orders_rev

,year,order_id,products,quantity,revenue,%_change
0,2014,969,1992,7579,483966.1261,NaN
1,2015,1038,2102,7979,470532.5090,-0.03
2,2016,1315,2587,9837,609205.5980,0.29
3,2017,1687,3312,12476,733215.2552,0.20


There has been a steady increase in number of orders placed, products sold and quantities. However, if you look at revenue, it decreased in 2015 by 3% but experienced a ~30% jump in 2016 and 20% jump in 2017

Yearly-revenue wise things look steady.

Looking at the revenue quarter wise to ensure this does not fall under Simpson's paradox. Hence, going a level deeper

In [36]:
# Total orders, sales, products, quantities: quarterly through 2014-2017
orders_rev_qtr = (orders_agg.groupby(["year", "quarter"])["order_id"].count().reset_index()).join(
    orders_agg.groupby(["year", "quarter"])["products"].sum(), 
    how="left", 
    on=["year", "quarter"]).join(
    orders_agg.groupby(["year", "quarter"])["quantity"].sum(), 
    how="left", 
    on=["year", "quarter"]).join(
    orders_agg.groupby(["year", "quarter"])["revenue"].sum(), 
    how="left", 
    on=["year", "quarter"])
orders_rev_qtr["%_change"] = round(orders_rev_qtr["revenue"].pct_change(),2)
orders_rev_qtr

,year,quarter,order_id,products,quantity,revenue,%_change
0,2014,1,185,385,1438,96498.7200,NaN
1,2014,2,208,405,1516,83355.5086,-0.14
2,2014,3,257,545,2097,139306.0173,0.67
3,2014,4,319,657,2528,164805.8802,0.18
4,2015,1,186,342,1301,90952.3496,-0.45
5,2015,2,248,488,1788,97852.8812,0.08
6,2015,3,275,588,2273,145554.2330,0.49
7,2015,4,329,684,2617,136173.0452,-0.06
8,2016,1,235,473,1782,136898.6390,0.01
9,2016,2,308,637,2394,149148.5428,0.09


In [32]:
print("Typical increase in revenue each quarter", round(orders_rev_qtr["%_change"].median(),2))

Typical increase in revenue each quarter 0.04


There seems to have been significant drop in revenue every third quarter as a repeating cycle through the years in comparison to the typical expected increase of ~ 4% every quarter.

Digging a level deeper - Month wise

In [98]:
# Total orders, sales, products, quantities: monthly through 2014-2017
orders_rev_month = round((orders_agg.groupby(["year", "month_num", "month"])["order_id"].count().reset_index()).join(
    orders_agg.groupby(["year", "month_num", "month"])["products"].sum(), 
    how="left", 
    on=["year", "month_num", "month"]).join(
    orders_agg.groupby(["year", "month_num", "month"])["quantity"].sum(), 
    how="left", 
    on=["year", "month_num", "month"]).join(
    orders_agg.groupby(["year", "month_num", "month"])["revenue"].sum(), 
    how="left", 
    on=["year", "month_num", "month"]),2).sort_values(["month_num", "year"]).pivot(
    index=["month_num","month"],
    columns = "year",
    values = "revenue"
).reset_index()
#orders_rev_month["%_change_rev"] = round(orders_rev_month["revenue"].pct_change(),2)
orders_rev_month.columns = ["month_num", "month", "2014", "2015", "2016", "2017"]
orders_rev_month.index = orders_rev_month["month"]
round(orders_rev_month[[ "2014", "2015", "2016", "2017"]].pct_change(axis=1)*100,2)

,2014,2015,2016,2017
month,,,,
January,NaN,1.36,29.65,70.14
February,NaN,62.66,137.54,1.57
March,NaN,-25.41,21.37,50.72
April,NaN,55.79,18.75,-13.54
May,NaN,4.37,110.01,-37.07
June,NaN,-1.45,35.10,22.44
July,NaN,-18.71,48.88,27.14
August,NaN,32.33,-7.49,63.30
September,NaN,0.94,-37.08,76.64


In [112]:
print("Typically every year for each month the orders place is:", round(orders_rev_month.groupby("year")["order_id"].median(), 2))

KeyError: 'year'

In [ ]:
print("Typically every year for each month the quantities sold is:", round(orders_agg.groupby("year")["quantity"].median()))

Typically every year for each month the quantities sold is: year
2014    8.0
2015    8.0
2016    7.0
2017    7.0
Name: quantity, dtype: float64


In [ ]:
print("Typically every year for each month the order value is:", round(orders_agg.groupby("year")["order_value"].median()))

Typically every year for each month the order value is: year
2014    504.0
2015    427.0
2016    463.0
2017    433.0
Name: order_value, dtype: float64


After digging a level deeper, it was observed that -on the products, quantities and orders level it seems like a steady growth.

But on the other hand, as the years go by the typical increase in monthly revenue has significantly dropped from 7% to ~2%.

And, the typical* monthly order value for each year has seen a slight drop. Indicating the company is selling more (in reference to orders placed, quantities & products sold had been steadily increasing) but still earning less revenue per order.

*Median order value was used instead of average order value to reduce the influence of unusually large orders

Products driving the most revenue - Pareto analysis

#### Hypothesis 1: Median Order Value (AOV) is declining because customers buy cheaper products

In [116]:
orders_agg["revenue"].sum()/orders_agg["order_id"].count()

np.float64(458.55849237372735)

In [119]:
print("The typical order value is:", round( orders_agg["revenue"].median(),2))

The typical order value is: 151.96


In [121]:
round(orders_agg.groupby("year")["revenue"].median(), 2)

year
2014    155.37
2015    162.07
2016    145.50
2017    148.26
Name: revenue, dtype: float64

In [122]:
round(orders_agg.groupby(["month_num","month","year"])["revenue"].median(), 2).reset_index()

,month_num,month,year,revenue
0,1,January,2014,222.29
1,1,January,2015,139.42
2,1,January,2016,126.18
3,1,January,2017,200.91
4,2,February,2014,90.61
5,2,February,2015,148.76
6,2,February,2016,178.97
7,2,February,2017,159.73
8,3,March,2014,154.21
9,3,March,2015,136.92


In [123]:
round(orders_agg.groupby(["year","quarter"])["revenue"].median(), 2).reset_index()

,year,quarter,revenue
0,2014,1,133.13
1,2014,2,138.38
2,2014,3,145.57
3,2014,4,194.32
4,2015,1,138.17
5,2015,2,146.36
6,2015,3,167.86
7,2015,4,182.91
8,2016,1,167.89
9,2016,2,159.39


#### Hypothesis 2: There are specific categories that are causing revenue decline

#### Hypothesis 3: There has been an increase in discount provided across products

#### Hypothesis 4: There has been a decline in revenue due to the underperformance of certain regions

#### Hypothesis 5: The reasoning for revenue decline is due to certain customer segments generating lower order values

In [12]:
pareto_product_analysis = pd.DataFrame(round(orders.groupby("Product ID")["Sales"].sum(),2).sort_values(ascending = False).reset_index())
pareto_product_analysis["cumulative_revenue"] = round(pareto_product_analysis["Sales"].cumsum(), 2)
pareto_product_analysis["cumulative_%"] = round((pareto_product_analysis["cumulative_revenue"]/
                                           pareto_product_analysis["Sales"].sum())*100,2)
pareto_product_analysis[pareto_product_analysis["cumulative_%"]<=80]


,Product ID,Sales,cumulative_revenue,cumulative_%
0,TEC-CO-10004722,61599.82,61599.82,2.68
1,OFF-BI-10003527,27453.38,89053.20,3.88
2,TEC-MA-10002412,22638.48,111691.68,4.86
3,FUR-CH-10002024,21870.58,133562.26,5.81
4,OFF-BI-10001359,19823.48,153385.74,6.68
...,...,...,...,...
408,TEC-MA-10002178,1391.40,1831425.02,79.73
409,TEC-PH-10004434,1386.69,1832811.71,79.79
410,TEC-PH-10001750,1385.79,1834197.50,79.85
411,OFF-AP-10002311,1383.08,1835580.58,79.91


It can be observed that around 22% of top products (based on revenue) contribute to 80% of revenue

How are different regions performing in terms of revenue

In [23]:
region_sales = orders[["Customer ID", 
        "Order ID", "Order Date",
        "Sales"]].merge(customer, 
                        on="Customer ID").merge(dates, left_on="Order Date",right_on="date").groupby(
                            ["year","Region"]
                        )["Sales"].sum().reset_index().pivot(
    index = "year",
    columns=["Region"], 
    values = "Sales").reset_index()
region_sales["%_change_central"] = round(region_sales["Central"].pct_change()*100, 2)
region_sales["%_change_east"] = round(region_sales["East"].pct_change()*100, 2)
region_sales["%_change_south"] = round(region_sales["South"].pct_change()*100, 2)
region_sales["%_change_west"] = round(region_sales["West"].pct_change()*100, 2)
region_sales[["year","Central","%_change_central","South","%_change_south","West","%_change_west","East","%_change_east"]]

Region,year,Central,%_change_central,South,%_change_south,West,%_change_west,East,%_change_east
0,2014,7.664723e+05,NaN,598561.0715,NaN,1.161716e+06,NaN,9.245633e+05,NaN
1,2015,8.038060e+05,4.87,569601.0226,-4.84,1.065228e+06,-8.31,9.100310e+05,-1.57
2,2016,1.105688e+06,37.56,687610.2686,20.72,1.487948e+06,39.68,1.189814e+06,30.74
3,2017,1.177444e+06,6.49,850383.8028,23.67,1.590504e+06,6.89,1.498019e+06,25.90


It can be observed that there had been a major drop in revenue in Central region from 37.56% to 6.49% in year 2017

Identifying which customers buy the most

In [ ]:
customer_sales = round(orders.groupby("Customer ID")["Sales"].sum(),2).sort_values(ascending = False).reset_index()
customer_sales["pct_contribution"] = round(customer_sales['Sales'] *100/ customer_sales['Sales'].sum(),2)
customer_sales

Monthly sales performance

In [ ]:
monthly_perf_2017 = orders_agg.groupby(["year","month_num","month"])["revenue"].sum().reset_index()
monthly_perf_2017["%_change_rev"] = round(monthly_perf_2017["revenue"].pct_change(),2)
monthly_perf_2017